# CVPR 2026 — Understanding the *concepts*, not just the outputs

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.

Six experiments, each revealing a mechanism the whole conference is built on. For each: what it is, the code, and the *aha*.

*If Colab asks to "Restart runtime" after the setup cell, click it, then continue from the next cell.*


In [ ]:
# Setup
!pip -q install diffusers transformers accelerate scikit-learn matplotlib timm
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg','img.jpg')
IMG = Image.open('img.jpg').convert('RGB')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev); IMG


## 1. Watch diffusion *think* (Generation)
Generation isn't one-shot — an image is built by repeatedly **denoising** pure noise. We capture the image at each step. **Aha:** structure emerges coarse→fine; this iterative denoiser is the engine behind image, video, and 3D generation.


In [ ]:
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16).to(dev)
steps=[]
def grab(p, i, t, kw):
    lat = kw['latents']
    img = p.vae.decode(lat.to(p.vae.dtype)/p.vae.config.scaling_factor, return_dict=False)[0]
    img = ((img/2+0.5).clamp(0,1)[0].permute(1,2,0).float().cpu().numpy()*255).astype('uint8')
    steps.append(img); return kw
_ = pipe('a red fox in a snowy forest, cinematic', num_inference_steps=6, guidance_scale=0.0,
         callback_on_step_end=grab, callback_on_step_end_tensor_inputs=['latents'])
fig,ax=plt.subplots(1,len(steps),figsize=(3*len(steps),3))
for k,(a,im) in enumerate(zip(ax,steps)): a.imshow(im); a.set_title(f'step {k+1}'); a.axis('off')
plt.suptitle('Diffusion denoising trajectory: noise -> image'); plt.show()


## 2. Objects segment themselves — with **no labels** (Self-supervised learning)
DINOv2 was trained on raw images with no annotations. We take its per-patch features and PCA them to RGB. **Aha:** distinct objects/parts light up as distinct colors — structure *emerges* from self-supervision. This is why SSL underpins modern vision.


In [ ]:
from transformers import AutoImageProcessor, AutoModel
from sklearn.decomposition import PCA
proc = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dino = AutoModel.from_pretrained('facebook/dinov2-base').to(dev).eval()
inp = proc(IMG, return_tensors='pt').to(dev)
with torch.no_grad(): out = dino(**inp)
feats = out.last_hidden_state[0,1:].float().cpu().numpy()
gh=inp['pixel_values'].shape[2]//14; gw=inp['pixel_values'].shape[3]//14; feats=feats[:gh*gw]
p = PCA(3).fit_transform(feats); p=(p-p.min(0))/(p.max(0)-p.min(0)+1e-6)
vis = Image.fromarray((p.reshape(gh,gw,3)*255).astype('uint8')).resize(IMG.size, Image.NEAREST)
fig,ax=plt.subplots(1,2,figsize=(10,5)); ax[0].imshow(IMG); ax[0].set_title('input'); ax[1].imshow(vis)
ax[1].set_title('DINOv2 features (PCA->RGB) — no labels used'); [a.axis('off') for a in ax]; plt.show()


## 3. Vision + language in **one shared space** (Vision-Language / CLIP)
CLIP maps images and text into the same space. We classify the image against *arbitrary* text labels — no training. **Aha:** because pictures and words share a space, you get *open-vocabulary* recognition and image↔text retrieval for free. This is the backbone of the VLM theme.


In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dev).eval()
cproc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
labels = ['a photo of a dog','a photo of a cat','a photo of a car','a snowy mountain','a plate of food']
ci = cproc(text=labels, images=IMG, return_tensors='pt', padding=True).to(dev)
with torch.no_grad(): probs = clip(**ci).logits_per_image.softmax(-1)[0].cpu().numpy()
plt.barh(labels, probs); plt.gca().invert_yaxis(); plt.title('CLIP zero-shot: no training, any labels'); plt.xlabel('probability'); plt.show()
for l,pr in sorted(zip(labels,probs), key=lambda x:-x[1]): print(f'{pr:5.1%}  {l}')


## 4. Where the model *looks* (Interpretability / hallucination)
We overlay a Vision Transformer's attention (the [CLS] token's attention to image patches). **Aha:** you can see *what the model attends to* — and when it attends to the wrong region, that's exactly the failure the 'hallucination-mitigation' papers target.


In [ ]:
from transformers import ViTModel, ViTImageProcessor
vp = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit = ViTModel.from_pretrained('google/vit-base-patch16-224', attn_implementation='eager').to(dev).eval()
vi = vp(IMG, return_tensors='pt').to(dev)
with torch.no_grad(): att = vit(**vi, output_attentions=True).attentions[-1]  # last layer
a = att[0].mean(0)[0,1:]  # CLS -> patches, avg heads
g = int(a.shape[0]**0.5); amap = a.reshape(g,g).float().cpu().numpy()
amap = (amap-amap.min())/(amap.max()-amap.min()+1e-6)
heat = Image.fromarray((plt.get_cmap('jet')(amap)[:,:,:3]*255).astype('uint8')).resize(IMG.size)
blend = Image.blend(IMG.resize(heat.size).convert('RGB'), heat, 0.55)
fig,ax=plt.subplots(1,2,figsize=(10,5)); ax[0].imshow(IMG); ax[1].imshow(blend)
ax[0].set_title('input'); ax[1].set_title('ViT attention (where it looks)'); [x.axis('off') for x in ax]; plt.show()


## 5. Fool the model with an **invisible** change (Adversarial robustness)
We add a tiny perturbation (FGSM) — imperceptible to you — and the classifier flips to a confident wrong answer. **Aha:** this fragility is why a whole CVPR subfield exists on robustness/security.


In [ ]:
import torchvision, torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
w=ResNet50_Weights.IMAGENET1K_V2; net=resnet50(weights=w).to(dev).eval(); cats=w.meta['categories']
tf=T.Compose([T.Resize(232),T.CenterCrop(224),T.ToTensor()])
x=tf(IMG).unsqueeze(0).to(dev); mean=torch.tensor([0.485,0.456,0.406],device=dev).view(1,3,1,1); std=torch.tensor([0.229,0.224,0.225],device=dev).view(1,3,1,1)
def pred(z):
    with torch.no_grad(): o=net((z-mean)/std).softmax(-1)[0]; i=o.argmax().item(); return cats[i], o[i].item()
c0,p0=pred(x)
x.requires_grad_(True); out=net((x-mean)/std); loss=out.max(); loss.backward()
xadv=(x+0.02*x.grad.sign()).clamp(0,1).detach()
c1,p1=pred(xadv)
fig,ax=plt.subplots(1,3,figsize=(13,4))
ax[0].imshow(x[0].permute(1,2,0).detach().cpu()); ax[0].set_title(f'original\n{c0} {p0:.0%}')
d=(xadv-x)[0].permute(1,2,0).detach().cpu().numpy(); d=(d-d.min())/(d.max()-d.min()+1e-6)
ax[1].imshow(d); ax[1].set_title('perturbation (amplified)')
ax[2].imshow(xadv[0].permute(1,2,0).cpu()); ax[2].set_title(f'attacked\n{c1} {p1:.0%}')
[a.axis('off') for a in ax]; plt.show()


## 6. Detect things described **in words** (Open-vocabulary perception)
No fixed class list — we hand the detector free-text queries. **Aha:** language-driven detection means you can find *anything you can describe*, the modern perception paradigm.


In [ ]:
from transformers import Owlv2Processor, Owlv2ForObjectDetection
op=Owlv2Processor.from_pretrained('google/owlv2-base-patch16-ensemble')
od=Owlv2ForObjectDetection.from_pretrained('google/owlv2-base-patch16-ensemble').to(dev).eval()
queries=['a dog','a hedge','grass','the dog\'s ear']
oi=op(text=[queries], images=IMG, return_tensors='pt').to(dev)
with torch.no_grad(): oo=od(**oi)
tgt=torch.tensor([IMG.size[::-1]]).to(dev)
res=op.post_process_object_detection(oo, threshold=0.25, target_sizes=tgt)[0]
import matplotlib.patches as pch
fig,ax=plt.subplots(figsize=(7,7)); ax.imshow(IMG)
for b,s,l in zip(res['boxes'],res['scores'],res['labels']):
    b=b.tolist(); ax.add_patch(pch.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],fill=False,color='lime',lw=2))
    ax.text(b[0],b[1]-4,f'{queries[l]} {s:.2f}',color='black',backgroundcolor='lime',fontsize=9)
ax.axis('off'); ax.set_title('Open-vocabulary detection from text queries'); plt.show()
